# Gatekeep — Phase 2: real-model baseline (Colab + Ollama)

Runs the Gatekeep benchmark against real tool-calling models (`qwen3:8b`, `llama3.1:8b`) served by Ollama on a Colab GPU.
Use **Runtime → Change runtime type → T4 GPU** (or better).

Rules: real models only (`sim:*` output is **not** a finding), and export summaries/reports only — never raw transcripts.
Results live on Google Drive, so the full-run cell can be re-run after a disconnect and it resumes (finished chunks are skipped).

In [ ]:
# 1. GPU check
import subprocess, sys
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "No GPU. Runtime -> Change runtime type -> GPU."

In [ ]:
# 2. Config + Drive (results persist across disconnects)
import os
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/gatekeep/phase2"
os.makedirs(OUT, exist_ok=True)
MODELS = ["qwen3:8b", "llama3.1:8b"]
POSTURES = ["none", "advisory", "enforced"]
REPEATS = 3
SEED = 0
WORKERS = 1   # raise to 2-4 only after checking GPU throughput
REPO = "https://github.com/jellaharshith/Gatekeep"
BRANCH = "main"
print(OUT)

In [ ]:
# 3. Install Gatekeep
!rm -rf /content/Gatekeep && git clone --depth 1 -b {BRANCH} {REPO} /content/Gatekeep
%cd /content/Gatekeep
!pip install -q -e .
!gatekeep --version

In [ ]:
# 4. Install and start Ollama, pull the models
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time, urllib.request
srv = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama did not start; see /content/ollama.log")
for m in MODELS:
    !ollama pull {m}

In [ ]:
# 5. Tool-calling smoke test. Bad tool calls would distort ASR and utility, so fail early.
import json, urllib.request
tool = {"type": "function", "function": {"name": "get_alert", "description": "Fetch an alert by id",
        "parameters": {"type": "object", "properties": {"alert_id": {"type": "string"}}, "required": ["alert_id"]}}}
for m in MODELS:
    body = {"model": m, "temperature": 0, "tools": [tool], "tool_choice": "auto",
            "messages": [{"role": "user", "content": "Look up alert A-1 using the tool."}]}
    req = urllib.request.Request("http://localhost:11434/v1/chat/completions", json.dumps(body).encode(),
                                 {"Content-Type": "application/json"})
    msg = json.load(urllib.request.urlopen(req, timeout=300))["choices"][0]["message"]
    calls = msg.get("tool_calls") or []
    ok = bool(calls)
    if ok:
        try: json.loads(calls[0]["function"]["arguments"])
        except Exception: ok = False
    print(f"{m}: {'OK' if ok else 'FAIL'}", calls[:1] or msg.get("content", "")[:200])
    assert ok, f"{m} did not return a valid tool call; results would be unreliable"

In [ ]:
# 6. Dry run: trial counts
models_csv = ",".join(f"ollama:{m}" for m in MODELS)
postures_csv = ",".join(POSTURES)
!gatekeep run --models {models_csv} --postures {postures_csv} --repeats {REPEATS} --seed {SEED} --dry-run

In [ ]:
# 7. Pilot (--sample 60), one model per run. Look at the numbers and the error count before scaling.
for m in MODELS:
    tag = m.replace(":", "-")
    !gatekeep run --models ollama:{m} --sample 60 --seed {SEED} --temperature 0 --workers {WORKERS} --out {OUT}/pilot-{tag}

In [ ]:
# 8. Full run in resumable chunks (one per model x posture). Re-run this cell after a disconnect;
#    chunks that already have summary.json are skipped. (If Colab restarted, re-run cells 2-4 first.)
import os, subprocess
for m in MODELS:
    for p in POSTURES:
        chunk = f"{OUT}/full/{m.replace(':', '-')}__{p}"
        if os.path.exists(f"{chunk}/summary.json"):
            print("skip", chunk); continue
        print("run ", chunk)
        subprocess.run(["gatekeep", "run", "--models", f"ollama:{m}", "--postures", p, "--repeats", str(REPEATS),
                        "--seed", str(SEED), "--temperature", "0", "--workers", str(WORKERS), "--out", chunk], check=True)

In [ ]:
# 9. Merge chunks and build the report (ASR +/- CI, clean utility, held/trial, paired p-values)
import glob, os
merged = f"{OUT}/full/merged"
os.makedirs(merged, exist_ok=True)
files = sorted(glob.glob(f"{OUT}/full/*__*/results.jsonl"))
print(len(files), "chunks")
with open(f"{merged}/results.jsonl", "w") as w:
    for f in files:
        w.write(open(f).read())
!gatekeep report {merged}/results.jsonl --out {merged}
from IPython.display import Markdown
Markdown(open(f"{merged}/report.md").read())

In [ ]:
# 10. Export summaries only (no raw transcripts), then download
import shutil, os
exp = "/content/phase2-export"; shutil.rmtree(exp, ignore_errors=True); os.makedirs(exp)
shutil.copy(f"{merged}/summary.json", exp); shutil.copy(f"{merged}/report.md", exp)
shutil.make_archive("/content/phase2-results", "zip", exp)
from google.colab import files as colab_files
colab_files.download("/content/phase2-results.zip")
# Then locally: unzip into results/ and commit summary.json + report.md.